# Multimodal Biosignal-Based Immersion Level Classification

Binary classification of user immersion states using EEG, GSR, and PPG signals.
- Model: EEGNet + 1D CNN encoders with cross-modal attention fusion
- Loss: BCEWithLogitsLoss
- Optimizer: Adam (lr=1e-3), Early stopping (patience=20)

## 1. Setup & Data Loading

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import os
from Dataset import BiosignalDataset
from model import FusionModel
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ─── Config ───────────────────────────────────────────────
batch_size = 8
epochs     = 50
lr         = 1e-3
patience   = 20
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

data_dir  = r"D:\2025-1capstoneProject\DATA_SLICED"   # update path
split_dir = r"D:\2025-1capstoneProject\data_splits"   # update path

train_dataset = BiosignalDataset(data_dir, split='train', split_path=split_dir)
val_dataset   = BiosignalDataset(data_dir, split='val',   split_path=split_dir)
test_dataset  = BiosignalDataset(data_dir, split='test',  split_path=split_dir)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32,         shuffle=False, num_workers=0, pin_memory=True)

print("Train:", len(train_dataset), "Val:", len(val_dataset), "Test:", len(test_dataset))

[INFO] train: 79968 samples loaded into memory ✅
[INFO] val: 7995 samples loaded into memory ✅
[INFO] test: 15996 samples loaded into memory ✅
Train: 79968 Val: 7995 Test: 15996


## 2. Model & Loss

In [2]:
model     = FusionModel().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

## 3. Training Loop

In [3]:
best_val_loss = float('inf')
train_losses, val_losses = [], []
train_accs,   val_accs   = [], []
early_stop_counter = 0

for epoch in range(1, epochs + 1):
    # ── Train ──
    model.train()
    train_loss, train_correct, total = 0, 0, 0
    for eeg, gsr, ppg, label in train_loader:
        eeg, gsr, ppg, label = (eeg.to(device), gsr.to(device),
                                 ppg.to(device), label.to(device).unsqueeze(1))
        optimizer.zero_grad()
        logits, _ = model(eeg, gsr, ppg)
        loss = criterion(logits, label)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        pred = (torch.sigmoid(logits) > 0.5).float()
        train_correct += (pred == label).sum().item()
        total += label.size(0)
    train_acc = train_correct / total
    train_losses.append(train_loss / len(train_loader))
    train_accs.append(train_acc)

    # ── Validation ──
    model.eval()
    val_loss, val_correct, total = 0, 0, 0
    with torch.no_grad():
        for eeg, gsr, ppg, label in val_loader:
            eeg, gsr, ppg, label = (eeg.to(device), gsr.to(device),
                                     ppg.to(device), label.to(device).unsqueeze(1))
            logits, _ = model(eeg, gsr, ppg)
            loss = criterion(logits, label)
            val_loss += loss.item()
            pred = (torch.sigmoid(logits) > 0.5).float()
            val_correct += (pred == label).sum().item()
            total += label.size(0)
    val_acc = val_correct / total
    val_losses.append(val_loss / len(val_loader))
    val_accs.append(val_acc)

    print(f"[Epoch {epoch}] Train Loss: {train_losses[-1]:.4f}, Acc: {train_acc:.4f} | "
          f"Val Loss: {val_losses[-1]:.4f}, Acc: {val_acc:.4f}")

    # ── Early stopping ──
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        early_stop_counter = 0
        torch.save(model.state_dict(), 'best_model.pt')
    else:
        early_stop_counter += 1
        if early_stop_counter >= patience:
            print("⏹️ Early stopping triggered.")
            break

[Epoch 1] Train Loss: 0.6892, Acc: 0.5406 | Val Loss: 0.6778, Acc: 0.5965
[Epoch 2] Train Loss: 0.6747, Acc: 0.5811 | Val Loss: 0.6449, Acc: 0.6396
[Epoch 3] Train Loss: 0.6384, Acc: 0.6257 | Val Loss: 0.5918, Acc: 0.6627
[Epoch 4] Train Loss: 0.6041, Acc: 0.6541 | Val Loss: 0.5895, Acc: 0.6629
[Epoch 5] Train Loss: 0.5777, Acc: 0.6757 | Val Loss: 0.5260, Acc: 0.7174
[Epoch 6] Train Loss: 0.5565, Acc: 0.6923 | Val Loss: 0.5349, Acc: 0.6981
[Epoch 7] Train Loss: 0.5417, Acc: 0.7013 | Val Loss: 0.5020, Acc: 0.7109
[Epoch 8] Train Loss: 0.5304, Acc: 0.7028 | Val Loss: 0.4935, Acc: 0.7232
[Epoch 9] Train Loss: 0.5180, Acc: 0.7111 | Val Loss: 0.4816, Acc: 0.7352
[Epoch 10] Train Loss: 0.5092, Acc: 0.7188 | Val Loss: 0.4916, Acc: 0.7323
[Epoch 11] Train Loss: 0.4989, Acc: 0.7287 | Val Loss: 0.4639, Acc: 0.7471
[Epoch 12] Train Loss: 0.4918, Acc: 0.7307 | Val Loss: 0.4527, Acc: 0.7457
[Epoch 13] Train Loss: 0.4846, Acc: 0.7324 | Val Loss: 0.4568, Acc: 0.7485
[Epoch 14] Train Loss: 0.4812, Acc

## 4. Learning Curves

In [4]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train')
plt.plot(val_losses,   label='Val')
plt.title('Loss'); plt.xlabel('Epoch'); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Train')
plt.plot(val_accs,   label='Val')
plt.title('Accuracy'); plt.xlabel('Epoch'); plt.legend()

plt.tight_layout()
plt.show()

## 5. Test Evaluation

In [5]:
print("💾  Test samples :", len(test_dataset))
model.load_state_dict(torch.load("best_model.pt", weights_only=True))
model.eval()

y_true, y_prob = [], []
with torch.no_grad():
    for eeg, gsr, ppg, label in test_loader:
        eeg, gsr, ppg = eeg.to(device), gsr.to(device), ppg.to(device)
        logits, _ = model(eeg, gsr, ppg)
        prob = torch.sigmoid(logits)
        y_prob.extend(prob.squeeze().cpu().numpy())
        y_true.extend(label.numpy())

y_pred = (np.array(y_prob) >= 0.5).astype(int)
print("\n✅ Accuracy :",        accuracy_score(y_true, y_pred))
print("\n✅ Confusion-matrix\n", confusion_matrix(y_true, y_pred))
print("\n✅ Detailed report\n",  classification_report(y_true, y_pred, digits=4))

💾  Test samples : 15996

✅ Accuracy : 0.7914478619654913

✅ Confusion-matrix
 [[7333 1100]
 [2236 5327]]

✅ Detailed report
               precision    recall  f1-score   support

         0.0     0.7663    0.8696    0.8147      8433
         1.0     0.8288    0.7044    0.7615      7563

    accuracy                         0.7914     15996
   macro avg     0.7976    0.7870    0.7881     15996
weighted avg     0.7959    0.7914    0.7896     15996
